# 09 - Batch processing

**Audience:** N participant CSVs to one summary DataFrame.

Demonstrates the full `BatchProcessor` workflow against any directory
of recordings. The example targets the directory containing the
discovered CSV; redirect by setting `BATCH_DIR` in cell 1 of section 1.

## What you'll get

- Discover CSVs in a directory.
- Configure the per-file pipeline (which metrics, parallelism).
- Add a custom analyzer (per-participant LHIPA-by-phase).
- Run the batch and view the summary DataFrame.
- Aggregate across participants.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Discover the batch

Defaults to the directory of the discovered CSV. Override `BATCH_DIR`
below to point at any folder of EyeTracking_*.csv files.

In [ ]:
BATCH_DIR = ctx.csv_path.parent
all_csvs = sorted(
    p for p in BATCH_DIR.glob("EyeTracking_*.csv")
    if not (p.stem.endswith("_SceneState") or p.stem.endswith("_SceneEvents"))
)
print(f"Batch directory: {BATCH_DIR}")
print(f"Main CSVs:       {len(all_csvs)}")
for p in all_csvs[:10]:
    print(f"  {p.name}")

## 2. Configure the pipeline

Disable LHIPA if PyWavelets isn't installed; it'll fail per-file silently
but slows the batch. Multi-process is recommended for >5 files.

In [ ]:
proc = ela.BatchProcessor(
    n_workers=1,            # bump to 4-8 for large batches
    show_progress=False,    # set True if tqdm is installed
    compute_lhipa=True,
    compute_entropy=True,
    compute_fixations=True,
)

## 3. Add a custom analyzer

Computes per-phase LHIPA so each row in the summary carries one column
per phase. Custom-analyzer exceptions are caught per-file and don't
fail the batch.

In [ ]:
def per_phase_lhipa(d):
    out = {}
    df = d.df
    if "phase" not in df.columns:
        return out
    have_pupil = "left_pupil_diameter" in df.columns or "right_pupil_diameter" in df.columns
    if not have_pupil:
        return out
    sr = d.get_sample_rate() or 90.0
    for phase, sub in df.groupby("phase"):
        if len(sub) < int(5 * sr):
            continue
        col = "left_pupil_diameter" if "left_pupil_diameter" in sub.columns else "right_pupil_diameter"
        pupil = sub[col].to_numpy(dtype=float)
        r = ela.calculate_lhipa(pupil, sample_rate=sr)
        if r.is_valid:
            out[f"lhipa_{phase}"] = r.lhipa
    return out
proc.add_analyzer(per_phase_lhipa)

## 4. Run

In [ ]:
if not all_csvs:
    print("No CSVs to process.")
else:
    results = proc.process_files(all_csvs, validate=False)
    summary = ela.BatchProcessor.results_to_dataframe(results)
    print(f"Processed: {len(results)} files")
    print(f"Successful: {sum(1 for r in results if r.success)}")
    print(f"Failed:     {sum(1 for r in results if not r.success)}")
    summary

## 5. Aggregate

In [ ]:
if not all_csvs:
    pass
else:
    agg = ela.BatchProcessor.summarize_results(results)
    for k, v in agg.items():
        if isinstance(v, float):
            print(f"  {k:<28} {v:.3f}")
        else:
            print(f"  {k:<28} {v}")

## 6. Save

Standard pattern: write the summary to disk for downstream stats.

In [ ]:
if all_csvs:
    out = BATCH_DIR / "batch_summary.csv"
    summary.to_csv(out, index=False)
    print(f"Wrote {out}")